# ADS-B coverage

Without a recorded off-block, the first sustained ground movement seen in ADS-B is the closest thing to it. The
question is whether `MVT − movement start` can stand in for the label. That needs two things: the aircraft must be
seen before it takes off, and the movement must start close to the stand.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
FEATURES = ROOT / "data/03_features"
COLUMNS = ["PHASE_mvt", "ADEP_mvt", "MVT_TIME_UTC_mvt", "SCHED_TIME_UTC_mvt", "adsb_first_observation_to_mvt_sec",
           "adsb_movement_to_mvt_sec", "adsb_movement_to_gate_direct_m"]


def load(name, year=None):
    frame = pd.read_parquet(FEATURES / name, columns=COLUMNS + (["TAXITIME_SEC_mvt"] if year else []))
    frame = frame.loc[frame["PHASE_mvt"].eq("DEP")]
    if year:
        frame = frame.loc[pd.to_datetime(frame["SCHED_TIME_UTC_mvt"], utc=True).dt.year.eq(year)]
    frame["month"] = pd.to_datetime(frame["MVT_TIME_UTC_mvt"], utc=True).dt.month
    # a ground movement counts only if it starts before the take-off
    frame["before take-off"] = frame["adsb_movement_to_mvt_sec"] > 0
    return frame


train = load("train_DEP_merged_sorted.parquet", year=2025)
final = load("final_ranking.parquet")
print(f"{len(train):,} departures in 2025, {len(final):,} in the final ranking file")

2,085,041 departures in 2025, 670,790 in the final ranking file


### When is the aircraft first seen?

Time of the first ADS-B position relative to the take-off, 2025 (percent of the departures per airport).

In [2]:
first = train["adsb_first_observation_to_mvt_sec"]
seen = pd.cut(first, [-np.inf, 0, 60, 300, np.inf],
              labels=["after take-off", "0–1 min before", "1–5 min before", "more than 5 min before"])
seen = seen.cat.add_categories("not seen").fillna("not seen")
(pd.crosstab(train["ADEP_mvt"], seen, normalize="index") * 100).round(0)

adsb_first_observation_to_mvt_sec,after take-off,0–1 min before,1–5 min before,more than 5 min before,not seen
ADEP_mvt,,,,,
EDDF,9.0,7.0,21.0,60.0,3.0
EDDM,16.0,1.0,8.0,72.0,3.0
EGLL,77.0,2.0,5.0,14.0,2.0
EHAM,1.0,1.0,12.0,82.0,4.0
LEBL,3.0,3.0,16.0,75.0,3.0
LEMD,25.0,3.0,11.0,50.0,11.0
LFPG,94.0,0.0,2.0,1.0,3.0
LIRF,46.0,8.0,13.0,18.0,15.0
LSZH,8.0,1.0,17.0,70.0,4.0


At LFPG and EGLL the receivers rarely see aircraft on the ground: most departures appear only after the take-off.
At LTFM there are almost no positions at all. At EHAM, LEBL, LSZH, EDDF and EDDM most aircraft are seen several
minutes before the take-off.

### Ground movement before take-off, per airport and month

Share of departures whose first sustained movement starts before the take-off, 2025.

In [3]:
(pd.crosstab(train["ADEP_mvt"], train["month"], values=train["before take-off"], aggfunc="mean") * 100).round(0)

month,1,2,3,4,5,6,7,8,9,10,11,12
ADEP_mvt,,,,,,,,,,,,
EDDF,82.0,91.0,91.0,91.0,89.0,87.0,84.0,83.0,88.0,84.0,92.0,92.0
EDDM,93.0,87.0,87.0,82.0,91.0,66.0,37.0,79.0,84.0,93.0,95.0,94.0
EGLL,41.0,40.0,22.0,3.0,4.0,7.0,9.0,14.0,14.0,20.0,34.0,36.0
EHAM,96.0,97.0,97.0,87.0,97.0,96.0,95.0,95.0,95.0,95.0,95.0,95.0
LEBL,95.0,97.0,98.0,97.0,98.0,96.0,88.0,94.0,90.0,93.0,95.0,95.0
LEMD,0.0,1.0,71.0,92.0,95.0,94.0,16.0,17.0,85.0,94.0,94.0,95.0
LFPG,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2.0,2.0,6.0,15.0,6.0
LIRF,0.0,0.0,0.0,0.0,20.0,71.0,88.0,81.0,28.0,42.0,43.0,64.0
LSZH,88.0,89.0,89.0,89.0,89.0,88.0,85.0,85.0,86.0,88.0,88.0,92.0


The same for the final ranking file (January, February, June and July 2026).

In [4]:
(pd.crosstab(final["ADEP_mvt"], final["month"], values=final["before take-off"], aggfunc="mean") * 100).round(0)

month,1,2,6,7
ADEP_mvt,,,,
EDDF,67.0,34.0,58.0,51.0
EDDM,94.0,95.0,92.0,93.0
EGLL,61.0,54.0,91.0,92.0
EHAM,94.0,95.0,91.0,92.0
LEBL,93.0,94.0,91.0,92.0
LEMD,95.0,95.0,75.0,58.0
LFPG,4.0,1.0,6.0,4.0
LIRF,62.0,53.0,1.0,4.0
LSZH,90.0,93.0,86.0,88.0


### ADS-B estimate against the label

Only departures with a movement before the take-off and a label up to one hour, split by whether the movement starts
within 100 m of the stand.

In [5]:
s = train.loc[train["before take-off"] & (train["TAXITIME_SEC_mvt"] <= 3600)].copy()
s["error"] = s["adsb_movement_to_mvt_sec"] - s["TAXITIME_SEC_mvt"]
s["near stand"] = s["adsb_movement_to_gate_direct_m"] < 100
s.groupby(["ADEP_mvt", "near stand"])["error"].agg(n="size", median="median",
                                                   rmse=lambda e: np.sqrt((e ** 2).mean())).round(0).unstack()

n            median           rmse       
near stand     False     True    False  True    False  True 
ADEP_mvt                                                    
EDDF        167874.0   33914.0  -469.0  -47.0   574.0  217.0
EDDM         56688.0   78925.0  -330.0  -45.0   475.0  207.0
EGLL         38157.0    9607.0  -932.0 -335.0  1076.0  391.0
EHAM        103806.0  131621.0  -275.0  -16.0   403.0  217.0
LEBL        115126.0   54939.0  -445.0  -32.0   568.0  213.0
LEMD        103931.0   31069.0  -521.0 -132.0   738.0  289.0
LFPG          6679.0      49.0  -868.0 -240.0  1346.0  233.0
LIRF         57797.0    3304.0  -742.0 -186.0   949.0  425.0
LSZH         76170.0   42380.0  -359.0  -28.0   523.0  195.0
LTFM           274.0       NaN  6420.0    NaN  8219.0    NaN

### What follows for the models

Whether ADS-B shows the taxi depends on the airport and changes from month to month, presumably with the receivers
that feed ADSB.lol. At EHAM, LEBL, LSZH, EDDM and EDDF the movement before take-off is seen for most departures; at
LFPG and LTFM almost never; at EGLL, LEMD and LIRF it comes and goes. In the final ranking LIRF has hardly any ground
movement in June and July 2026 (1 % and 4 %, against 71 % and 88 % in 2025), and EDDF only 34 % in February.

Even when the movement is seen before the take-off, it usually starts away from the stand, and the estimate is then
clearly too short. Only near the stand is it close to the label. The measured ADS-B time is therefore not submitted as
the prediction for a flight, and it is not a safe lower bound either (`adsb_lower_bound.ipynb`). Its values enter the
models as features (time from the first movement and from the first position to the take-off, distance of the movement
start to the stand and to the runway), and the models learn how far off the measurement typically is. 2025 contains the
same kinds of gaps, so the models also learn what a late or missing sighting means.